## Importing Libraries

In [18]:
import pandas as pd
import spacy
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score


## Loading the true and fake datasets and merging them into one

In [2]:
fake_df = pd.read_csv("/kaggle/input/fake-and-real-news-dataset/Fake.csv")
true_df = pd.read_csv("/kaggle/input/fake-and-real-news-dataset/True.csv")

# Add a label column: 0 for fake news, 1 for true news
fake_df["label"] = 0
true_df["label"] = 1

# Merge both datasets into a single DataFrame
data = pd.concat([fake_df, true_df], ignore_index=True)

data.head()

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


## Let's inspect some basic info about the dataset

In [3]:
# Display basic info
print(data.info())

# Check for missing values
print(data.isnull().sum())

# Check class distribution
print(data["label"].value_counts())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44898 non-null  object
 1   text     44898 non-null  object
 2   subject  44898 non-null  object
 3   date     44898 non-null  object
 4   label    44898 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.7+ MB
None
title      0
text       0
subject    0
date       0
label      0
dtype: int64
label
0    23481
1    21417
Name: count, dtype: int64


### We can see there is no null values and that the number of true and fake news are almost equally shared.

## We have two columns which are subject and date which are irrelevant in identifying if the news are fake or not, hence we will remove them

In [4]:
data.drop(columns=["subject", "date"], inplace=True)

data.head()

,title,text,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,0


### We can also combine the title and text of the article, which might make it easier for the model to classify some news as true or fake

In [5]:
data["combined_text"] = data["title"] + " " + data["text"]

data.drop(columns=["title", "text"], inplace=True) # we can drop the title and text columns now


data.head()

,label,combined_text
0,0,Donald Trump Sends Out Embarrassing New Year’...
1,0,Drunk Bragging Trump Staffer Started Russian ...
2,0,Sheriff David Clarke Becomes An Internet Joke...
3,0,Trump Is So Obsessed He Even Has Obama’s Name...
4,0,Pope Francis Just Called Out Donald Trump Dur...


## Now that we have our data we can start with the preprocessing

In [10]:
# we will use spacy for the preprocessing of the data

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"]) # we load spacy's english language model, we try disabling the parser and NER as they might not be important for this task, so that we can also finish preprocessing faster

# Function to preprocess text using spaCy
def preprocess_text(text):
    doc = nlp(text.lower())  # First step, we use Case folding, to make all text lowercase
    
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct] # we tokenize the text and remove stopping words as well as applying lemmatization
    return " ".join(tokens)

## Applying preprocessing to data

In [11]:
# Now we apply the preprocessing to our combined text and add it as a column in our dataframe
tqdm.pandas(desc="Preprocessing Text")
data['preprocessed_text'] = data['combined_text'].progress_apply(preprocess_text)

data.head()

Preprocessing Text: 100%|██████████| 44898/44898 [25:08<00:00, 29.76it/s]


,label,combined_text,preprocessed_text
0,0,Donald Trump Sends Out Embarrassing New Year’...,donald trump send embarrass new year eve mes...
1,0,Drunk Bragging Trump Staffer Started Russian ...,drunk bragging trump staffer start russian c...
2,0,Sheriff David Clarke Becomes An Internet Joke...,sheriff david clarke internet joke threaten ...
3,0,Trump Is So Obsessed He Even Has Obama’s Name...,trump obsessed obama code website image chri...
4,0,Pope Francis Just Called Out Donald Trump Dur...,pope francis call donald trump christmas spe...


## Vectorization

In [14]:
# Now that we finished preprocessing our data, we can go ahead and vectorize it, we will use TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)  # we limit the max number of features/words to 5000 for computational efficiency, and increase if there is the need.
X = vectorizer.fit_transform(data['preprocessed_text']) #our vectors for each document are stored in X


## Splitting data to train and test set

In [19]:
# label column (0 for fake news, 1 for true news)
y = data['label']

# Splitting the data into training and testing sets (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Training with Naive Bayes

In [20]:
# Now that we have our data ready and splitted properly we can go ahead and train the model
nb = MultinomialNB()
nb.fit(X_train, y_train)

MultinomialNB()

## Testing accuracy

In [26]:
y_pred = nb.predict(X_test) # we predict on the test set

# Last step, we calculate the accuracy score
print(f'Accuracy: {accuracy_score(y_test, y_pred)*100}')

Accuracy: 94.26503340757239
